<a href="https://colab.research.google.com/github/danilo576/Cuda/blob/main/CUDA_Paralelni_sistemi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUDA — Paralelni sistemi

Materijal i vežbe iz programiranja na GPU-u pomoću NVIDIA CUDA tehnologije.

# ⚙️ 1. Podešavanje okruženja

In [ ]:
!pip install -q nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

## 1.1 Provera GPU-a

In [ ]:
!nvidia-smi

## 1.2 Provera CUDA kompajlera

In [ ]:
!nvcc --version

# 🚀 2. Prvi CUDA program

Pre nego što krenemo sa vežbama, napravićemo jednostavan CUDA program koji izvršava funkciju na GPU-u.

In [ ]:
%%cuda

#include <stdio.h>

__global__ void hello() {
    printf("Hello from GPU!\n");
}

int main() {
    hello<<<1, 1>>>();
    cudaDeviceSynchronize();

    return 0;
}

Hello from GPU!



## 2.1 CPU i GPU

In [ ]:
%%cuda

#include <stdio.h>

__global__ void hello_from_gpu() {
    printf("Hello from GPU!\n");
}

int main() {

    printf("Hello from CPU!\n");

    hello_from_gpu<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Hello from CPU!
Hello from GPU!



## 2.2 Blokovi i threadovi

In [ ]:
%%cuda

#include <stdio.h>

__global__ void hello() {

    printf(
        "Block: %d, Thread: %d\n",
        blockIdx.x,
        threadIdx.x
    );
}

int main() {

    hello<<<2, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}

Block: 0, Thread: 0
Block: 0, Thread: 1
Block: 0, Thread: 2
Block: 0, Thread: 3
Block: 1, Thread: 0
Block: 1, Thread: 1
Block: 1, Thread: 2
Block: 1, Thread: 3



# 🧮 3. Sabiranje vektora

Sabiranje dva vektora korišćenjem CUDA kernela.

Za svaki element važi:

C[i] = A[i] + B[i]

In [ ]:
%%cuda

#include <stdio.h>

#define N 10

__global__ void vector_add(
    const int *A,
    const int *B,
    int *C
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {

    int A[N], B[N], C[N];

    for (int i = 0; i < N; i++) {
        A[i] = i;
        B[i] = i * 2;
    }

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, N * sizeof(int));
    cudaMalloc(&d_B, N * sizeof(int));
    cudaMalloc(&d_C, N * sizeof(int));

    cudaMemcpy(
        d_A, A,
        N * sizeof(int),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_B, B,
        N * sizeof(int),
        cudaMemcpyHostToDevice
    );

    vector_add<<<1, N>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    cudaMemcpy(
        C, d_C,
        N * sizeof(int),
        cudaMemcpyDeviceToHost
    );

    for (int i = 0; i < N; i++) {
        printf(
            "C[%d] = %d\n",
            i,
            C[i]
        );
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

C[0] = 0
C[1] = 3
C[2] = 6
C[3] = 9
C[4] = 12
C[5] = 15
C[6] = 18
C[7] = 21
C[8] = 24
C[9] = 27

